In [ ]:
import os
import re
import subprocess

# ==========================================================================
# M5 HARDENED KAGGLE DUAL-T4 WORKER LAUNCHER (Pinned SHA + Kaggle Secrets)
# 3 Explicit Worker Profiles (select via KAGGLE_WORKER_PROFILE):
#   - 'qa_dual_gpu' (default): GPU 0 = Qwen/Qwen3-8B (NF4, base instruct),
#                              GPU 1 = BAAI/bge-m3 + bge-reranker-v2-m3 + mDeBERTa-v3 NLI
#   - 'ingest':                Dedicated OCR/Layout/Embedding ingest phase (skips Qwen 8B)
#   - '32b_tp2':               GPU 0+1 TP=2 Qwen/Qwen3-32B-AWQ via vLLM
# ==========================================================================

# 1. Pin BOTH boot.sh download URL AND repository checkout to an exact 40-char commit SHA
GIT_COMMIT_SHA = os.environ.get(
    "GIT_COMMIT_SHA",
    "ffff517aa9df6c3da84002e8dfbd2c2cb17e7b53",  # Update to your released M5 commit SHA
).strip()
assert re.fullmatch(r"[0-9a-fA-F]{40}", GIT_COMMIT_SHA), (
    f"GIT_COMMIT_SHA must be a 40-char hex commit SHA, got: {GIT_COMMIT_SHA!r}"
)
os.environ["GIT_COMMIT_SHA"] = GIT_COMMIT_SHA
os.environ["REQUIRE_PINNED_COMMIT"] = "true"

# 2. Worker profile & pinned model configuration
os.environ["KAGGLE_WORKER_PROFILE"] = os.environ.get("KAGGLE_WORKER_PROFILE", "qa_dual_gpu")
os.environ["BASE_QWEN_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["GPU_MAX_MODEL_ID"] = "Qwen/Qwen3-32B-AWQ"
os.environ["QWEN_USE_BASE_INSTRUCT"] = "true"
os.environ["SKIP_DRIVE_LORA_BUNDLE"] = "true"
os.environ["PRELOAD_RETRIEVAL_MODELS"] = "true"
os.environ["PREFER_HF_SOTA_NLI"] = "true"
os.environ["FACTCHECK_MODEL_ID"] = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
os.environ["ALLOW_HF_SOTA_FALLBACK"] = "true"
os.environ["PUBLIC_TUNNEL"] = "true"
os.environ["SEPARATE_ADMIN_INFERENCE_TOKENS"] = "true"

# 3. Load DISTINCT Admin & Inference tokens strictly from Kaggle Secrets (never hardcoded or printed)
def _load_kaggle_secret(secret_name: str) -> bool:
    if os.environ.get(secret_name):
        return True
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(secret_name)
        if isinstance(val, str) and val.strip():
            os.environ[secret_name] = val.strip()
            return True
    except Exception:
        return False
    return False

_has_admin = _load_kaggle_secret("SUPERVISOR_ADMIN_TOKEN")
_has_api = _load_kaggle_secret("SUPERVISOR_API_TOKEN")
if _has_admin and _has_api and os.environ["SUPERVISOR_ADMIN_TOKEN"] == os.environ["SUPERVISOR_API_TOKEN"]:
    raise RuntimeError("SUPERVISOR_ADMIN_TOKEN and SUPERVISOR_API_TOKEN must be distinct secrets!")
print(f"[Auth] Kaggle Secrets loaded -> admin={_has_admin}, inference={_has_api} (values redacted)")

# 4. Fetch boot.sh from the EXACT pinned commit SHA URL (never mutable branch refs)
BOOT_URL = f"https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/{GIT_COMMIT_SHA}/kaggle/boot.sh"
print(f"[Bootstrap] Fetching pinned boot.sh @ {GIT_COMMIT_SHA}")
subprocess.run(["bash", "-c", f"curl -fsSL '{BOOT_URL}' | bash"], check=True)
